<a href="https://colab.research.google.com/github/alimoorreza/CS181-fall26-notes/blob/main/day07_cnn_with_activation_and_pooling_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS181: Computer Vision, Fall 2026**
### Convoluational Neural Network (CNN) Components

![MLP architecture](https://raw.githubusercontent.com/alimoorreza/CS181-fall26-notes/main/images/cnn.drawio.png)


In [2]:
# import torch library
import torch
import numpy as np
import torch.nn as nn

## **Question 0.1: let's create a CNN with three conv2d layers and connect them in a sequence following the specifications below:**


The first convolution layer has:
  > its input volume has 1 channels. It can take any arbitrary volume with $(C_{in}=1, H_{in}=?, W_{in}=?)$. For example, you could create an input volume of $(C_{in}=q, H_{in}=50, W_{in}=50)$.

  > the output volume will be of 2 channels, ie, $(C_{out}=6$

  > each filter has a size of (3x3) with a __stride__ and a __padding__ as follows:
  - $F=3$
  - $S=2$
  - $P=0$


Following a similar convention, the second convolution layer has:
  > its input volume has 2 channels

  > the output volume will be of 4 channels

  > each filter has a size of (3x3) with a __stride__ and a __padding__ as follows:
  - $F=3$
  - $S=2$
  - $P=0$

The third convolution layer has:
  > its input volume has 4 channels

  > the output volume will be of 8 channels

  > each filter has a size of (3x3) with a __stride__ and a __padding__ as follows:
  - $F=3$
  - $S=2$
  - $P=0$



**After each convolution layer you should use the *ReLU()* activation function for its output**
<!--**Use a default stride size is 1 (don't need to change)**
**Use a default padding size is 0 (don't need to change)**-->
> [Reference: nn.Conv2d()](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)


In [3]:
my_cnn = nn.Sequential(
            nn.Conv2d(1, 2, 3, 2, 0),
            nn.ReLU(),
            nn.Conv2d(2, 4, 3, 2, 0),
            nn.ReLU(),
            nn.Conv2d(4, 8, 3, 2, 0),
            nn.ReLU()
)

## **Question 0.3: Generate a random input for your CNN.**

In [7]:
my_cnn

Sequential(
  (0): Conv2d(1, 2, kernel_size=(3, 3), stride=(2, 2))
  (1): ReLU()
  (2): Conv2d(2, 4, kernel_size=(3, 3), stride=(2, 2))
  (3): ReLU()
  (4): Conv2d(4, 8, kernel_size=(3, 3), stride=(2, 2))
  (5): ReLU()
)

In [4]:
torch.manual_seed(2) # for reproducibility (you will get the same random number every time you run this cell)
# Let's generate one random samples of (x1, x2, h, w) for the above linear network
# We want to create only one input sample
# ---> x1 = 1

# Recall that our network's first layer has 1 input channels. hence our input channel should have a value of 1
# ---> x2 = 1

# our network's first convolution layer can take any height and width of the input volume. let's use h=50 and w=50
# ---> h = 50
# ---> w = 50

# We can create a random matrix with a size of (1, 1, 50, 50). In other words we do the following:
# number_of_samples     = 1
# random_input_volume   = torch.randn(number_of_samples, num_of_channels, height, width)

random_input_volume     = torch.randn( (1, 1, 50, 50) )


print(f'input volume\'s shape: \n{random_input_volume.shape}\n')
#print(f'input numbers: \n{random_input_volume.numpy()}\n')



input volume's shape: 
torch.Size([1, 1, 50, 50])



## **Question 0.4: Perform a forward pass of the input volume through your CNN.**


In [5]:
# perform the forward pass through the network
output                = my_cnn(random_input_volume)
#print(f'output layer values: \n{output.data.numpy()}\n')
print(f'size of the final output (last layer): \n{output.data.shape}\n')



size of the final output (last layer): 
torch.Size([1, 8, 5, 5])



Alternatively, you could use an object-oriented programming approach to create a class for this network. This would make it easier to inspect the output volumes at the intermediate layers.


In [6]:
class MyCNN(nn.Module):

    def __init__(self):

        super(MyCNN, self).__init__()
        self.conv2d_layer1          = nn.Conv2d(1, 2, 3, 2, 0)
        self.conv2d_layer2          = nn.Conv2d(2, 4, 3, 2, 0)
        self.conv2d_layer3          = nn.Conv2d(4, 8, 3, 2, 0)


    def forward(self, x):

        print("shape of input (     ): ", x.shape)
        x = self.conv2d_layer1(x)
        print("output shape (conv2d-layer#1): ", x.shape)
        x = self.conv2d_layer2(x)
        print("output shape (conv2d-layer#2): ", x.shape)
        x = self.conv2d_layer3(x)
        print("output shape (conv2d-layer#3): ", x.shape)
        return x

my_cnn_standard = MyCNN()

In [7]:
output                = my_cnn_standard(random_input_volume)
#print(f'output layer values: \n{output.data.numpy()}\n')
print(f'Size of the final output (last layer): \n{output.data.shape}\n')


shape of input (     ):  torch.Size([1, 1, 50, 50])
output shape (conv2d-layer#1):  torch.Size([1, 2, 24, 24])
output shape (conv2d-layer#2):  torch.Size([1, 4, 11, 11])
output shape (conv2d-layer#3):  torch.Size([1, 8, 5, 5])
Size of the final output (last layer): 
torch.Size([1, 8, 5, 5])



## **Question 0.5: let's modify the above CNN by inserting $ReLU$ activation layer and a $max$-$pooling$ layer in between each pair of adjacent conv2d layers and connect them in a sequence following the specifications below:**



![MLP architecture](https://raw.githubusercontent.com/alimoorreza/CS181-fall26-notes/main/images/cnn_with_maxpooling.drawio.svg)


In [11]:
class MyCNNWithActivationPooling(nn.Module):

    def __init__(self):

        super(MyCNNWithActivationPooling, self).__init__()
        self.conv2d_layer1          = nn.Conv2d(1, 2, 3, 2, 0)
        self.relu1                  = nn.ReLU()
        self.maxpooling_layer1      = nn.MaxPool2d(2,2) # mention the filter size of the pooling

        self.conv2d_layer2          = nn.Conv2d(2, 4, 3, 2, 0)
        self.relu2                  = nn.ReLU()
        self.maxpooling_layer2      = nn.MaxPool2d(2,2) # mention the filter size of the pooling

        self.conv2d_layer3          = nn.Conv2d(4, 8, 3, 2, 0)
        self.relu3                  = nn.ReLU()
        self.maxpooling_layer3      = nn.MaxPool2d(2,2) # mention the filter size of the pooling



    def forward(self, x):

        # first conv2d + ReLU + maxpooling2d
        print("shape of input (     ): ", x.shape)
        x = self.conv2d_layer1(x)
        x = self.relu1(x)
        print("output shape (conv2d-layer#1): ", x.shape)
        x = self.maxpooling_layer1(x)
        print("output shape (maxpoolinglayer#1): ", x.shape)


        # second conv2d + ReLU + maxpooling2d
        x = self.conv2d_layer2(x)
        x = self.relu2(x)
        print("output shape (conv2d-layer#2): ", x.shape)
        x = self.maxpooling_layer2(x)
        print("output shape (maxpoolinglayer#2): ", x.shape)


        # third conv2d + ReLU + maxpooling2d
        x = self.conv2d_layer3(x)
        x = self.relu3(x)
        print("output shape (conv2d-layer#3): ", x.shape)
        x = self.maxpooling_layer3(x)
        print("output shape (maxpoolinglayer#3): ", x.shape)

        return x

my_cnn2 = MyCNNWithActivationPooling()

In [12]:
# create ONE random input volume

input = torch.randn( (1, 1, 100, 100) )

final_output = my_cnn2(input)
print(f"shape of output at the rightmost end of the network: {final_output.shape}")


shape of input (     ):  torch.Size([1, 1, 100, 100])
output shape (conv2d-layer#1):  torch.Size([1, 2, 49, 49])
output shape (maxpoolinglayer#1):  torch.Size([1, 2, 24, 24])
output shape (conv2d-layer#2):  torch.Size([1, 4, 11, 11])
output shape (maxpoolinglayer#2):  torch.Size([1, 4, 5, 5])
output shape (conv2d-layer#3):  torch.Size([1, 8, 2, 2])
output shape (maxpoolinglayer#3):  torch.Size([1, 8, 1, 1])
shape of output at the rightmost end of the network: torch.Size([1, 8, 1, 1])
